# CMGAN Training Notebook

Notebook version of the training logic from train.py.

This version also includes Learning without Forgetting (LwF) with configurable $\lambda_{LwF}$ and `n_reset` for the snapshot interval.

In [2]:
%pip install torchinfo natsort einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [einops]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import csv
import logging
import os
from collections import OrderedDict

import torch
import torch.nn.functional as F
import torchaudio
from torch.utils.data import DataLoader
from torchinfo import summary

from data import dataloader
from models import discriminator
from models.generator import TSCNet
from utils import power_compress, power_uncompress

logging.basicConfig(level=logging.INFO)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

ModuleNotFoundError: No module named 'torch'

In [ ]:
epochs = 120
batch_size = 4
log_interval = 500
decay_epoch = 30
init_lr = 5e-4
cut_len = 16000 * 2
data_dir = "/home/lugra002/speechbrain/data/cmgan_voicebank"
save_model_dir = "./saved_model"
metrics_csv_name = "train_metrics.csv"
loss_weights = [0.1, 0.9, 0.2, 0.05]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_workers = 2

# LwF settings (Li & Hoiem, 2017)
lwf_enabled = True
lambda_lwf = 1.0
lambda_lwf_decay = 1.0
# Optional linear schedule override (takes precedence over lambda_lwf/lambda_lwf_decay).
lambda_lwf_start = None
lambda_lwf_end = None
n_reset = 10
kd_temperature = 2.0

def load_data_notebook(ds_dir, batch_size, num_workers, cut_len):
    torchaudio.set_audio_backend("sox_io")
    train_dir = os.path.join(ds_dir, "train")
    test_dir = os.path.join(ds_dir, "test")

    train_ds = dataloader.DemandDataset(train_dir, cut_len)
    test_ds = dataloader.DemandDataset(test_dir, cut_len)

    train_loader = DataLoader(
        dataset=train_ds,
        batch_size=batch_size,
        pin_memory=torch.cuda.is_available(),
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
    )
    test_loader = DataLoader(
        dataset=test_ds,
        batch_size=batch_size,
        pin_memory=torch.cuda.is_available(),
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
    )

    return train_loader, test_loader

NameError: name 'torch' is not defined

In [ ]:
import os

# LwF ablations: vary the weight and the snapshot interval.
ABLATION_PRESETS = {
    "lwf_l0p5_n50": {
        "lwf_enabled": True,
        "lambda_lwf": 0.5,
        "lambda_lwf_decay": 1.0,
        "lambda_lwf_start": None,
        "lambda_lwf_end": None,
        "n_reset": 50,
    },
    "lwf_l0p5_n10": {
        "lwf_enabled": True,
        "lambda_lwf": 0.5,
        "lambda_lwf_decay": 1.0,
        "lambda_lwf_start": None,
        "lambda_lwf_end": None,
        "n_reset": 10,
    },
    "lwf_l0p7_n10": {
        "lwf_enabled": True,
        "lambda_lwf": 0.7,
        "lambda_lwf_decay": 1.0,
        "lambda_lwf_start": None,
        "lambda_lwf_end": None,
        "n_reset": 10,
    },
    "lwf_l0p7_n50": {
        "lwf_enabled": True,
        "lambda_lwf": 0.7,
        "lambda_lwf_decay": 1.0,
        "lambda_lwf_start": None,
        "lambda_lwf_end": None,
        "n_reset": 50,
    },
}

def apply_ablation_preset(name, results_root="./saved_model_ablations"):
    global lwf_enabled, lambda_lwf, lambda_lwf_decay, lambda_lwf_start, lambda_lwf_end, n_reset
    global save_model_dir, metrics_csv_name

    if name not in ABLATION_PRESETS:
        raise ValueError(f"Unknown preset '{name}'. Available: {list(ABLATION_PRESETS.keys())}")

    cfg = ABLATION_PRESETS[name]
    lwf_enabled = cfg["lwf_enabled"]
    lambda_lwf = cfg["lambda_lwf"]
    lambda_lwf_decay = cfg["lambda_lwf_decay"]
    lambda_lwf_start = cfg["lambda_lwf_start"]
    lambda_lwf_end = cfg["lambda_lwf_end"]
    n_reset = cfg["n_reset"]

    save_model_dir = os.path.join(results_root, name)
    metrics_csv_name = "train_metrics.csv"

    print("Applied ablation:")
    print(f"  preset          : {name}")
    print(f"  lwf_enabled     : {lwf_enabled}")
    print(f"  lambda_lwf      : {lambda_lwf}")
    print(f"  lambda_lwf_decay: {lambda_lwf_decay}")
    print(f"  lambda_start    : {lambda_lwf_start}")
    print(f"  lambda_end      : {lambda_lwf_end}")
    print(f"  n_reset         : {n_reset}")
    print(f"  save_model_dir  : {save_model_dir}")

print("Available presets:")
for preset_name in ABLATION_PRESETS:
    print(" -", preset_name)

Available presets:
 - baseline_no_lwf
 - lwf_l0p5_n10
 - lwf_l1p0_n10
 - lwf_l1p0_n50


## Ablation Presets

- `lwf_l0p5_n10`: LwF with lambda = 0.5 and `n_reset = 10`
- `lwf_l0p7_n10`: LwF with lambda = 0.7 and `n_reset = 10`
- `lwf_l0p5_n50`: LwF with lambda = 0.5 and `n_reset = 50`
- `lwf_l0p7_n50`: LwF with lambda = 0.7 and `n_reset = 50`

In [ ]:
class Trainer:
    def __init__(self, train_ds, test_ds):
        self.n_fft = 400
        self.hop = 100
        self.train_ds = train_ds
        self.test_ds = test_ds

        self.model = TSCNet(num_channel=64, num_features=self.n_fft // 2 + 1).to(device)
        try:
            summary(self.model, [(1, 2, cut_len // self.hop + 1, int(self.n_fft / 2) + 1)])
        except Exception as exc:
            logging.info("Model summary skipped: %s", exc)

        self.discriminator = discriminator.Discriminator(ndf=16).to(device)
        try:
            summary(
                self.discriminator,
                [
                    (1, 1, int(self.n_fft / 2) + 1, cut_len // self.hop + 1),
                    (1, 1, int(self.n_fft / 2) + 1, cut_len // self.hop + 1),
                ],
            )
        except Exception as exc:
            logging.info("Discriminator summary skipped: %s", exc)

        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=init_lr)
        self.optimizer_disc = torch.optim.AdamW(self.discriminator.parameters(), lr=2 * init_lr)
        self.metrics_csv_path = os.path.join(save_model_dir, metrics_csv_name)

        self.old_generator_state = None
        self.old_discriminator_state = None

    def lwf_active(self):
        if not lwf_enabled:
            return False

        if lambda_lwf_start is not None or lambda_lwf_end is not None:
            start_val = float(lambda_lwf_start) if lambda_lwf_start is not None else 0.0
            end_val = float(lambda_lwf_end) if lambda_lwf_end is not None else start_val
            return max(start_val, end_val) > 0.0

        return float(lambda_lwf) > 0.0

    def current_lambda_lwf(self, epoch):
        if not self.lwf_active():
            return 0.0

        total_epochs = max(int(epochs), 1)
        progress = 0.0 if total_epochs == 1 else float(epoch) / float(total_epochs - 1)

        if lambda_lwf_start is not None or lambda_lwf_end is not None:
            start_val = float(lambda_lwf_start) if lambda_lwf_start is not None else 0.0
            end_val = float(lambda_lwf_end) if lambda_lwf_end is not None else start_val
            return max(0.0, start_val + progress * (end_val - start_val))

        base_lambda = float(lambda_lwf)
        decay_factor = float(lambda_lwf_decay)
        if total_epochs == 1 or decay_factor == 1.0:
            return base_lambda

        decay_scale = 1.0 - (1.0 - decay_factor) * progress
        return max(0.0, base_lambda * decay_scale)

    def snapshot_module_state(self, module):
        return {
            "params": OrderedDict(
                (name, param.detach().clone())
                for name, param in module.named_parameters()
            ),
            "buffers": OrderedDict(
                (name, buf.detach().clone()) for name, buf in module.named_buffers()
            ),
        }

    def functional_call_module(self, module, params, buffers, *args, **kwargs):
        try:
            from torch.func import functional_call

            return functional_call(module, (params, buffers), args, kwargs)
        except (ImportError, TypeError):
            from torch.nn.utils.stateless import functional_call

            merged_state = OrderedDict(params)
            merged_state.update(buffers)
            return functional_call(module, merged_state, args, kwargs)

    def refresh_lwf_snapshots(self):
        self.old_generator_state = self.snapshot_module_state(self.model)
        self.old_discriminator_state = self.snapshot_module_state(self.discriminator)

    def maybe_refresh_lwf_snapshots(self, epoch):
        if not self.lwf_active():
            return

        if self.old_generator_state is None or self.old_discriminator_state is None:
            self.refresh_lwf_snapshots()
            return

        if int(n_reset) > 0 and epoch % int(n_reset) == 0:
            self.refresh_lwf_snapshots()

    def old_generator_forward(self, clean, noisy):
        c = torch.sqrt(noisy.size(-1) / torch.sum((noisy**2.0), dim=-1)).unsqueeze(-1)
        noisy = noisy * c
        clean = clean * c

        window = torch.hamming_window(self.n_fft, device=device)
        noisy_spec = torch.stft(noisy, self.n_fft, self.hop, window=window, onesided=True, return_complex=False)
        clean_spec = torch.stft(clean, self.n_fft, self.hop, window=window, onesided=True, return_complex=False)

        noisy_spec = power_compress(noisy_spec).permute(0, 1, 3, 2)
        clean_spec = power_compress(clean_spec)
        clean_real = clean_spec[:, 0, :, :].unsqueeze(1)
        clean_imag = clean_spec[:, 1, :, :].unsqueeze(1)

        snapshot = self.old_generator_state
        est_real, est_imag = self.functional_call_module(
            self.model,
            snapshot["params"],
            snapshot["buffers"],
            noisy_spec,
        )
        est_real = est_real.permute(0, 1, 3, 2)
        est_imag = est_imag.permute(0, 1, 3, 2)
        est_mag = torch.sqrt(est_real**2 + est_imag**2)
        clean_mag = torch.sqrt(clean_real**2 + clean_imag**2)

        est_spec_uncompress = power_uncompress(est_real, est_imag).squeeze(1)
        est_audio = torch.istft(
            torch.view_as_complex(est_spec_uncompress.contiguous()),
            self.n_fft,
            self.hop,
            window=window,
            onesided=True,
        )

        return {
            "est_real": est_real,
            "est_imag": est_imag,
            "est_mag": est_mag,
            "clean_mag": clean_mag,
            "est_audio": est_audio,
        }

    def old_discriminator_score(self, deg_spec, ref_spec):
        snapshot = self.old_discriminator_state
        return self.functional_call_module(
            self.discriminator,
            snapshot["params"],
            snapshot["buffers"],
            deg_spec,
            ref_spec,
        )

    def kd_loss(self, student_score, teacher_score, temperature=2.0):
        eps = 1e-8
        student_prob = torch.sigmoid(student_score / temperature)
        teacher_prob = torch.sigmoid(teacher_score / temperature)

        student_dist = torch.cat([student_prob, 1.0 - student_prob], dim=1)
        teacher_dist = torch.cat([teacher_prob, 1.0 - teacher_prob], dim=1)

        student_log = torch.log(student_dist.clamp(min=eps, max=1.0))
        teacher_dist = teacher_dist.clamp(min=eps, max=1.0)
        return F.kl_div(student_log, teacher_dist, reduction="batchmean") * (temperature**2)

    def forward_generator_step(self, clean, noisy):
        c = torch.sqrt(noisy.size(-1) / torch.sum((noisy**2.0), dim=-1)).unsqueeze(-1)
        noisy = noisy * c
        clean = clean * c

        window = torch.hamming_window(self.n_fft, device=device)
        noisy_spec = torch.stft(noisy, self.n_fft, self.hop, window=window, onesided=True, return_complex=False)
        clean_spec = torch.stft(clean, self.n_fft, self.hop, window=window, onesided=True, return_complex=False)

        noisy_spec = power_compress(noisy_spec).permute(0, 1, 3, 2)
        clean_spec = power_compress(clean_spec)
        clean_real = clean_spec[:, 0, :, :].unsqueeze(1)
        clean_imag = clean_spec[:, 1, :, :].unsqueeze(1)

        est_real, est_imag = self.model(noisy_spec)
        est_real = est_real.permute(0, 1, 3, 2)
        est_imag = est_imag.permute(0, 1, 3, 2)
        est_mag = torch.sqrt(est_real**2 + est_imag**2)
        clean_mag = torch.sqrt(clean_real**2 + clean_imag**2)

        est_spec_uncompress = power_uncompress(est_real, est_imag).squeeze(1)
        est_audio = torch.istft(
            torch.view_as_complex(est_spec_uncompress.contiguous()),
            self.n_fft,
            self.hop,
            window=window,
            onesided=True,
        )

        return {
            "est_real": est_real,
            "est_imag": est_imag,
            "est_mag": est_mag,
            "clean_real": clean_real,
            "clean_imag": clean_imag,
            "clean_mag": clean_mag,
            "est_audio": est_audio,
        }

    def generator_loss_components(self, generator_outputs, lwf_lambda=0.0, old_est_audio=None):
        predict_fake_metric = self.discriminator(generator_outputs["clean_mag"], generator_outputs["est_mag"])
        gen_loss_gan = F.mse_loss(predict_fake_metric.flatten(), generator_outputs["one_labels"].float())

        loss_mag = F.mse_loss(generator_outputs["est_mag"], generator_outputs["clean_mag"])
        loss_ri = F.mse_loss(generator_outputs["est_real"], generator_outputs["clean_real"]) + F.mse_loss(
            generator_outputs["est_imag"], generator_outputs["clean_imag"]
        )
        time_loss = torch.mean(torch.abs(generator_outputs["est_audio"] - generator_outputs["clean"]))

        lwf_gen_loss = torch.tensor(0.0, device=device)
        if lwf_lambda > 0.0 and old_est_audio is not None:
            lwf_gen_loss = F.mse_loss(generator_outputs["est_audio"], old_est_audio)

        total_loss = (
            loss_weights[0] * loss_ri
            + loss_weights[1] * loss_mag
            + loss_weights[2] * time_loss
            + loss_weights[3] * gen_loss_gan
            + lwf_lambda * lwf_gen_loss
        )

        return total_loss, {
            "loss_ri": loss_ri.detach().item(),
            "loss_mag": loss_mag.detach().item(),
            "time_loss": time_loss.detach().item(),
            "gen_loss_gan": gen_loss_gan.detach().item(),
            "lwf_gen_loss": lwf_gen_loss.detach().item(),
        }

    def calculate_generator_loss(self, generator_outputs):
        total_loss, _ = self.generator_loss_components(generator_outputs)
        return total_loss

    def calculate_discriminator_loss(self, generator_outputs, lwf_lambda=0.0):
        length = generator_outputs["est_audio"].size(-1)
        est_audio_list = list(generator_outputs["est_audio"].detach().cpu().numpy())
        clean_audio_list = list(generator_outputs["clean"].detach().cpu().numpy()[:, :length])
        pesq_score = discriminator.batch_pesq(clean_audio_list, est_audio_list)

        if pesq_score is None:
            return None, None, 0.0

        predict_enhance_metric = self.discriminator(generator_outputs["clean_mag"], generator_outputs["est_mag"].detach())
        predict_max_metric = self.discriminator(generator_outputs["clean_mag"], generator_outputs["clean_mag"])
        disc_loss = F.mse_loss(predict_max_metric.flatten(), generator_outputs["one_labels"]) + F.mse_loss(
            predict_enhance_metric.flatten(), pesq_score
        )

        lwf_disc_loss = torch.tensor(0.0, device=device)
        if lwf_lambda > 0.0 and self.old_discriminator_state is not None:
            with torch.no_grad():
                old_predict_enhance = self.old_discriminator_score(
                    generator_outputs["clean_mag"], generator_outputs["est_mag"].detach()
                ).detach()
                old_predict_max = self.old_discriminator_score(
                    generator_outputs["clean_mag"], generator_outputs["clean_mag"]
                ).detach()

            lwf_disc_loss = 0.5 * (
                self.kd_loss(predict_enhance_metric, old_predict_enhance, temperature=kd_temperature)
                + self.kd_loss(predict_max_metric, old_predict_max, temperature=kd_temperature)
            )
            disc_loss = disc_loss + lwf_lambda * lwf_disc_loss

        return disc_loss, float(pesq_score.detach().mean().item()), lwf_disc_loss.detach().item()

    def train_step(self, batch, epoch):
        clean = batch[0].to(device)
        noisy = batch[1].to(device)
        one_labels = torch.ones(clean.size(0), device=device)

        lwf_lambda = self.current_lambda_lwf(epoch)
        generator_outputs = self.forward_generator_step(clean, noisy)
        generator_outputs["one_labels"] = one_labels
        generator_outputs["clean"] = clean

        old_est_audio = None
        if lwf_lambda > 0.0 and self.old_generator_state is not None:
            with torch.no_grad():
                old_est_audio = self.old_generator_forward(clean, noisy)["est_audio"].detach()

        loss, g_stats = self.generator_loss_components(
            generator_outputs, lwf_lambda=lwf_lambda, old_est_audio=old_est_audio
        )
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        discrim_loss_metric, pesq_mean, lwf_disc_loss = self.calculate_discriminator_loss(
            generator_outputs, lwf_lambda=lwf_lambda
        )
        if discrim_loss_metric is not None:
            self.optimizer_disc.zero_grad()
            discrim_loss_metric.backward()
            self.optimizer_disc.step()
        else:
            discrim_loss_metric = torch.tensor([0.0], device=device)

        return {
            "gen_loss": loss.item(),
            "disc_loss": discrim_loss_metric.item(),
            "pesq_mean": pesq_mean,
            "lwf_lambda": lwf_lambda,
            "lwf_disc_loss": lwf_disc_loss,
            **g_stats,
        }

    @torch.no_grad()
    def test_step(self, batch):
        clean = batch[0].to(device)
        noisy = batch[1].to(device)
        one_labels = torch.ones(clean.size(0), device=device)

        generator_outputs = self.forward_generator_step(clean, noisy)
        generator_outputs["one_labels"] = one_labels
        generator_outputs["clean"] = clean

        loss, g_stats = self.generator_loss_components(generator_outputs, lwf_lambda=0.0, old_est_audio=None)
        discrim_loss_metric, pesq_mean, lwf_disc_loss = self.calculate_discriminator_loss(
            generator_outputs, lwf_lambda=0.0
        )
        if discrim_loss_metric is None:
            discrim_loss_metric = torch.tensor([0.0], device=device)

        return {
            "gen_loss": loss.item(),
            "disc_loss": discrim_loss_metric.item(),
            "pesq_mean": pesq_mean,
            "lwf_lambda": 0.0,
            "lwf_disc_loss": lwf_disc_loss,
            **g_stats,
        }

    def _init_epoch_tracker(self):
        return {
            "steps": 0,
            "gen_loss": 0.0,
            "disc_loss": 0.0,
            "loss_ri": 0.0,
            "loss_mag": 0.0,
            "time_loss": 0.0,
            "gen_loss_gan": 0.0,
            "lwf_gen_loss": 0.0,
            "lwf_disc_loss": 0.0,
            "lwf_lambda_sum": 0.0,
            "pesq_sum": 0.0,
            "pesq_count": 0,
        }

    def _update_epoch_tracker(self, tracker, step_stats):
        tracker["steps"] += 1
        tracker["gen_loss"] += step_stats["gen_loss"]
        tracker["disc_loss"] += step_stats["disc_loss"]
        tracker["loss_ri"] += step_stats["loss_ri"]
        tracker["loss_mag"] += step_stats["loss_mag"]
        tracker["time_loss"] += step_stats["time_loss"]
        tracker["gen_loss_gan"] += step_stats["gen_loss_gan"]
        tracker["lwf_gen_loss"] += step_stats["lwf_gen_loss"]
        tracker["lwf_disc_loss"] += step_stats["lwf_disc_loss"]
        tracker["lwf_lambda_sum"] += step_stats.get("lwf_lambda", 0.0)

        if step_stats["pesq_mean"] is not None:
            tracker["pesq_sum"] += step_stats["pesq_mean"]
            tracker["pesq_count"] += 1

    def _finalize_epoch_tracker(self, tracker):
        steps = max(tracker["steps"], 1)
        stats = {
            "gen_loss": tracker["gen_loss"] / steps,
            "disc_loss": tracker["disc_loss"] / steps,
            "loss_ri": tracker["loss_ri"] / steps,
            "loss_mag": tracker["loss_mag"] / steps,
            "time_loss": tracker["time_loss"] / steps,
            "gen_loss_gan": tracker["gen_loss_gan"] / steps,
            "lwf_gen_loss": tracker["lwf_gen_loss"] / steps,
            "lwf_disc_loss": tracker["lwf_disc_loss"] / steps,
            "lwf_lambda": tracker["lwf_lambda_sum"] / steps,
            "pesq_mean": None,
        }
        if tracker["pesq_count"] > 0:
            stats["pesq_mean"] = tracker["pesq_sum"] / tracker["pesq_count"]
        return stats

    def test(self):
        self.model.eval()
        self.discriminator.eval()

        tracker = self._init_epoch_tracker()
        for batch in self.test_ds:
            step_stats = self.test_step(batch)
            self._update_epoch_tracker(tracker, step_stats)

        stats = self._finalize_epoch_tracker(tracker)
        logging.info(
            "Val | gen: %.4f disc: %.4f ri: %.4f mag: %.4f time: %.4f gan: %.4f lwf_g: %.4f lwf_d: %.4f pesq: %s",
            stats["gen_loss"],
            stats["disc_loss"],
            stats["loss_ri"],
            stats["loss_mag"],
            stats["time_loss"],
            stats["gen_loss_gan"],
            stats["lwf_gen_loss"],
            stats["lwf_disc_loss"],
            "n/a" if stats["pesq_mean"] is None else f"{stats['pesq_mean']:.4f}",
        )
        return stats

    def _ensure_metrics_csv(self):
        if os.path.exists(self.metrics_csv_path):
            return

        os.makedirs(os.path.dirname(self.metrics_csv_path), exist_ok=True)
        with open(self.metrics_csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(
                [
                    "epoch",
                    "train_gen_loss",
                    "train_disc_loss",
                    "train_loss_ri",
                    "train_loss_mag",
                    "train_time_loss",
                    "train_gen_loss_gan",
                    "train_lwf_gen_loss",
                    "train_lwf_disc_loss",
                    "train_lwf_lambda",
                    "train_pesq_mean",
                    "val_gen_loss",
                    "val_disc_loss",
                    "val_loss_ri",
                    "val_loss_mag",
                    "val_time_loss",
                    "val_gen_loss_gan",
                    "val_lwf_gen_loss",
                    "val_lwf_disc_loss",
                    "val_lwf_lambda",
                    "val_pesq_mean",
                ]
            )

    def _append_metrics_csv(self, epoch, train_stats, val_stats):
        with open(self.metrics_csv_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(
                [
                    epoch,
                    train_stats["gen_loss"],
                    train_stats["disc_loss"],
                    train_stats["loss_ri"],
                    train_stats["loss_mag"],
                    train_stats["time_loss"],
                    train_stats["gen_loss_gan"],
                    train_stats["lwf_gen_loss"],
                    train_stats["lwf_disc_loss"],
                    train_stats["lwf_lambda"],
                    train_stats["pesq_mean"],
                    val_stats["gen_loss"],
                    val_stats["disc_loss"],
                    val_stats["loss_ri"],
                    val_stats["loss_mag"],
                    val_stats["time_loss"],
                    val_stats["gen_loss_gan"],
                    val_stats["lwf_gen_loss"],
                    val_stats["lwf_disc_loss"],
                    val_stats["lwf_lambda"],
                    val_stats["pesq_mean"],
                ]
            )

    def train(self):
        scheduler_G = torch.optim.lr_scheduler.StepLR(self.optimizer, step_size=decay_epoch, gamma=0.5)
        scheduler_D = torch.optim.lr_scheduler.StepLR(self.optimizer_disc, step_size=decay_epoch, gamma=0.5)
        os.makedirs(save_model_dir, exist_ok=True)
        self._ensure_metrics_csv()

        for epoch in range(epochs):
            self.maybe_refresh_lwf_snapshots(epoch)
            self.model.train()
            self.discriminator.train()

            train_tracker = self._init_epoch_tracker()
            for idx, batch in enumerate(self.train_ds):
                step = idx + 1
                step_stats = self.train_step(batch, epoch=epoch)
                self._update_epoch_tracker(train_tracker, step_stats)

                if step % log_interval == 0:
                    logging.info(
                        "Epoch %s Step %s | gen: %.4f disc: %.4f ri: %.4f mag: %.4f time: %.4f gan: %.4f lwf_g: %.4f lwf_d: %.4f lambda: %.4f pesq: %s",
                        epoch,
                        step,
                        step_stats["gen_loss"],
                        step_stats["disc_loss"],
                        step_stats["loss_ri"],
                        step_stats["loss_mag"],
                        step_stats["time_loss"],
                        step_stats["gen_loss_gan"],
                        step_stats["lwf_gen_loss"],
                        step_stats["lwf_disc_loss"],
                        step_stats["lwf_lambda"],
                        "n/a" if step_stats["pesq_mean"] is None else f"{step_stats['pesq_mean']:.4f}",
                    )

            train_stats = self._finalize_epoch_tracker(train_tracker)
            val_stats = self.test()

            logging.info(
                "Epoch %s Summary | train_gen: %.4f train_disc: %.4f val_gen: %.4f val_disc: %.4f lambda: %.4f",
                epoch,
                train_stats["gen_loss"],
                train_stats["disc_loss"],
                val_stats["gen_loss"],
                val_stats["disc_loss"],
                train_stats["lwf_lambda"],
            )

            self._append_metrics_csv(epoch, train_stats, val_stats)

            path = os.path.join(save_model_dir, "CMGAN_epoch_" + str(epoch) + "_" + str(val_stats["gen_loss"])[:5])
            torch.save(self.model.state_dict(), path)
            scheduler_G.step()
            scheduler_D.step()

NameError: name 'torch' is not defined

In [ ]:
train_ds, test_ds = load_data_notebook(data_dir, batch_size, num_workers, cut_len)
trainer = Trainer(train_ds, test_ds)
trainer

In [ ]:
# Run the LwF ablations one after another.
PILOT_ABLATION_ORDER = [
    "lwf_l0p5_n10",
    "lwf_l0p7_n10",
    "lwf_l0p5_n50",
    "lwf_l0p7_n50",
]

# Reduce the epoch count here for quick tests.
# epochs = 20

for preset_name in PILOT_ABLATION_ORDER:
    apply_ablation_preset(preset_name)
    train_ds, test_ds = load_data_notebook(data_dir, batch_size, num_workers, cut_len)
    trainer = Trainer(train_ds, test_ds)
    trainer.train()

Applied ablation:
  preset          : baseline_no_lwf
  lwf_enabled     : False
  lambda_lwf      : 0.0
  lambda_lwf_decay: 1.0
  lambda_start    : None
  lambda_end      : None
  n_reset         : 10
  save_model_dir  : ./saved_model_ablations/baseline_no_lwf


In [ ]:
# This cell is kept for manual single-run testing if needed.
# trainer.train()